Environment Setup

In [ ]:
# Cell [1] — Install all dependencies
!pip install pydantic great-expectations pandas apscheduler tenacity loguru pyarrow fastparquet -q

import importlib
for pkg in ["pydantic", "great_expectations", "apscheduler", "tenacity", "loguru", "pyarrow"]:
    try:
        mod = importlib.import_module(pkg)
        version = getattr(mod, '__version__', 'ok')
        print(f"✓ {pkg} {version}")
    except ImportError:
        print(f"✗ {pkg} NOT FOUND")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 10.9 MB/s eta 0:00:00
✓ pydantic 2.12.3
✓ great_expectations 1.18.0
✓ apscheduler 3.11.2
✓ tenacity ok
✓ loguru 0.7.3
✓ pyarrow 18.1.0


In [ ]:
# Cell [2] — Structured logging with Loguru + pipeline config
from loguru import logger
from dataclasses import dataclass, field
from pathlib import Path
import sys

# Remove default handler and configure custom format
logger.remove()
logger.add(
    sys.stdout,
    level="INFO",
    format="{time:HH:mm:ss} | {level: <8} | {message}",
    colorize=True
)
logger.add(
    "pipeline.log",
    rotation="10 MB",
    retention="7 days",
    level="DEBUG"
)


@dataclass
class PipelineConfig:
    """Central configuration for the pipeline. Override fields as needed."""
    source_path: Path = Path("data/raw")
    output_path: Path = Path("data/processed")
    schedule_cron: str = "0 6 * * *"   # 06:00 UTC daily
    max_retries: int = 3
    batch_size: int = 10_000
    alert_email: str = "ops@example.com"
    pass_rate_threshold: float = 0.95   # minimum valid-row ratio
    null_rate_limit: float = 0.05
    min_rows: int = 100


cfg = PipelineConfig()
logger.info(f"Config loaded — schedule: {cfg.schedule_cron} | output: {cfg.output_path}")


05:57:23 | INFO     | Config loaded — schedule: 0 6 * * * | output: data/processed


Data Ingestion

In [ ]:
# Cell [3] — Pluggable ingestor with strategy pattern
from abc import ABC, abstractmethod
import pandas as pd
import numpy as np


class BaseIngestor(ABC):
    """All ingestors return a raw DataFrame. Implement read() for each source."""

    @abstractmethod
    def read(self) -> pd.DataFrame:
        ...


class CsvIngestor(BaseIngestor):
    """Reads a CSV file from disk. Swap for S3/GCS path as needed."""

    def __init__(self, path: str):
        self.path = path

    def read(self) -> pd.DataFrame:
        df = pd.read_csv(self.path)
        logger.info(f"Ingested {len(df):,} rows from {self.path}")
        return df


class SyntheticIngestor(BaseIngestor):
    """Generates synthetic transaction data for demo/testing.
    Deliberately inserts nulls and negative amounts to exercise validation.
    """

    def __init__(self, n: int = 1_000, seed: int = 42):
        self.n = n
        self.seed = seed

    def read(self) -> pd.DataFrame:
        rng = np.random.default_rng(self.seed)
        n = self.n

        amounts = rng.normal(100, 30, n).round(2)
        # Inject ~1% negative amounts to test validation
        neg_idx = rng.choice(n, size=int(n * 0.01), replace=False)
        amounts[neg_idx] = -abs(amounts[neg_idx])

        df = pd.DataFrame({
            "id": range(n),
            "amount": amounts,
            "category": rng.choice(["A", "B", "C", None], n, p=[0.4, 0.3, 0.2, 0.1]),
            "ts": pd.date_range("2024-01-01", periods=n, freq="1h"),
        })

        logger.info(f"Generated {n:,} synthetic rows (seed={self.seed})")
        return df


# ── Run ingestion
raw_df = SyntheticIngestor(n=1_000).read()

print(f"\nShape: {raw_df.shape}")
print(f"Null counts:\n{raw_df.isnull().sum()}")
raw_df.head()

05:57:48 | INFO     | Generated 1,000 synthetic rows (seed=42)

Shape: (1000, 4)
Null counts:
id            0
amount        0
category    110
ts            0
dtype: int64


,id,amount,category,ts
0,0,109.14,A,2024-01-01 00:00:00
1,1,68.80,A,2024-01-01 01:00:00
2,2,122.51,C,2024-01-01 02:00:00
3,3,128.22,C,2024-01-01 03:00:00
4,4,41.47,A,2024-01-01 04:00:00


Schema & Data Validation

In [ ]:
# Cell [4] — Row-level schema validation with Pydantic v2
from pydantic import BaseModel, field_validator, ValidationError
from typing import Optional, Literal
from datetime import datetime


class TransactionRecord(BaseModel):
    """Pydantic model defining the expected schema for each transaction row."""

    id: int
    amount: float
    category: Optional[Literal["A", "B", "C"]]
    ts: datetime

    @field_validator("amount")
    @classmethod
    def amount_positive(cls, v: float) -> float:
        if v <= 0:
            raise ValueError(f"amount must be positive, got {v:.2f}")
        return v

    @field_validator("id")
    @classmethod
    def id_non_negative(cls, v: int) -> int:
        if v < 0:
            raise ValueError(f"id must be non-negative, got {v}")
        return v


def validate_batch(
    df: pd.DataFrame,
    pass_threshold: float = 0.95
) -> tuple[pd.DataFrame, list[dict]]:
    """Validate every row against TransactionRecord.

    Returns:
        valid_df: DataFrame of passing rows
        errors:   list of {row_id, field, message} dicts
    """
    valid_rows, errors = [], []

    for row in df.to_dict("records"):
        try:
            parsed = TransactionRecord(**row)
            valid_rows.append(parsed.model_dump())
        except ValidationError as e:
            for err in e.errors():
                errors.append({
                    "row_id": row.get("id"),
                    "field": err["loc"][0] if err["loc"] else "unknown",
                    "message": err["msg"],
                })

    pass_rate = len(valid_rows) / len(df)
    logger.info(
        f"Validation: {len(valid_rows):,} valid | {len(errors):,} errors | "
        f"{pass_rate*100:.1f}% pass rate"
    )

    if pass_rate < pass_threshold:
        logger.warning(
            f"Pass rate {pass_rate*100:.1f}% is below threshold "
            f"{pass_threshold*100:.0f}% — investigate upstream data"
        )

    valid_df = pd.DataFrame(valid_rows) if valid_rows else pd.DataFrame(columns=df.columns)
    return valid_df, errors


# ── Run validation
valid_df, validation_errors = validate_batch(raw_df, cfg.pass_rate_threshold)

print(f"\nValid rows: {len(valid_df):,}")
print(f"Validation errors: {len(validation_errors)}")
if validation_errors:
    print("\nSample errors:")
    for e in validation_errors[:5]:
        print(f"  row {e['row_id']} | {e['field']}: {e['message']}")


05:58:19 | INFO     | Validation: 989 valid | 11 errors | 98.9% pass rate

Valid rows: 989
Validation errors: 11

Sample errors:
  row 125 | amount: Value error, amount must be positive, got -110.71
  row 187 | amount: Value error, amount must be positive, got -63.27
  row 258 | amount: Value error, amount must be positive, got -95.99
  row 340 | amount: Value error, amount must be positive, got -31.26
  row 399 | amount: Value error, amount must be positive, got -78.86


In [ ]:
import great_expectations as gx
import pandas as pd


def build_ge_suite(df: pd.DataFrame) -> dict:
    """Run a Great Expectations expectation suite against a DataFrame.

    Returns a results summary dict with passed/evaluated counts.
    """
    context = gx.get_context(mode="ephemeral")

    # Register data source
    ds = context.data_sources.add_pandas("pandas_source")
    da = ds.add_dataframe_asset("transactions")
    batch_def = da.add_batch_definition_whole_dataframe("full_batch")
    batch = batch_def.get_batch(batch_parameters={"dataframe": df})

    # Build expectation suite
    suite = context.suites.add(gx.ExpectationSuite(name="txn_suite"))

    suite.add_expectation(
        gx.expectations.ExpectColumnToExist(column="amount"))
    suite.add_expectation(
        gx.expectations.ExpectColumnToExist(column="id"))
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToBeBetween(
            column="amount", min_value=0, max_value=300))
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToNotBeNull(column="id"))
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToNotBeNull(column="ts"))
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToBeInSet(
            column="category",
            value_set=["A", "B", "C"],
            mostly=0.90))   # allow up to 10% nulls / unknowns
    suite.add_expectation(
        gx.expectations.ExpectColumnValuesToBeUnique(column="id"))

    # Run validation
    validator = context.get_validator(
        batch=batch,
        expectation_suite=suite
    )
    results = validator.validate()
    stats = results.statistics

    # The 'logger' variable is not defined in the current scope. Assuming it's defined globally or in an outer scope.
    # For this fix, I'll assume `logger` is available or should be imported/defined. Let's add a placeholder if not.
    # If you intend to use `logger` from a previous cell, ensure it's imported or declared before this cell executes.
    # For now, I'll include a placeholder import if it's missing, or rely on existing global `logger` if it was from earlier cells.
    try:
        logger.info(
            f"GE suite: {stats['successful_expectations']}/"
            f"{stats['evaluated_expectations']} expectations passed"
        )
    except NameError:
        print(f"GE suite: {stats['successful_expectations']}/" +
              f"{stats['evaluated_expectations']} expectations passed")

    if not results.success:
        failed = [
            r.expectation_config.type
            for r in results.results
            if not r.success
        ]
        try:
            logger.warning(f"Failed expectations: {failed}")
        except NameError:
            print(f"WARNING: Failed expectations: {failed}")

    return {
        "success": results.success,
        "passed": stats["successful_expectations"],
        "evaluated": stats["evaluated_expectations"],
    }


# Assuming `valid_df` is defined in a previous cell.
# Assuming `logger` is defined in a previous cell.
# ── Run GE suite
ge_results = build_ge_suite(valid_df)
print(f"\nGE Suite passed: {ge_results['success']}")
print(f"Expectations: {ge_results['passed']}/{ge_results['evaluated']}")

INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmp69ilmh5_' for ephemeral docs site


Calculating Metrics:   0%|          | 0/25 [00:00<?, ?it/s]

06:00:11 | INFO     | GE suite: 6/6 expectations passed

GE Suite passed: True
Expectations: 6/6


Transformations

In [ ]:
# Cell [6] — Composable, testable transform functions
from functools import reduce
from typing import Callable

# Type alias for transform functions
TransformFn = Callable[[pd.DataFrame], pd.DataFrame]


def fill_category(df: pd.DataFrame) -> pd.DataFrame:
    """Replace null categories with 'UNKNOWN'."""
    return df.assign(category=df["category"].fillna("UNKNOWN"))


def add_amount_tier(df: pd.DataFrame) -> pd.DataFrame:
    """Bin amount into labelled tiers: low / medium / high / premium."""
    bins = [0, 50, 100, 150, float("inf")]
    labels = ["low", "medium", "high", "premium"]
    return df.assign(
        tier=pd.cut(df["amount"], bins=bins, labels=labels).astype(str)
    )


def extract_date_parts(df: pd.DataFrame) -> pd.DataFrame:
    """Extract date, hour, and day-of-week from the timestamp column."""
    ts = pd.to_datetime(df["ts"])
    return df.assign(
        date=ts.dt.date,
        hour=ts.dt.hour,
        day_of_week=ts.dt.day_name(),
    )


def normalize_amount(df: pd.DataFrame) -> pd.DataFrame:
    """Z-score normalize the amount column."""
    mu, sigma = df["amount"].mean(), df["amount"].std()
    return df.assign(amount_z=((df["amount"] - mu) / sigma).round(4))


def drop_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    """Remove duplicate IDs, keeping the first occurrence."""
    before = len(df)
    df = df.drop_duplicates(subset=["id"], keep="first")
    dropped = before - len(df)
    if dropped:
        logger.warning(f"Dropped {dropped} duplicate rows")
    return df


def apply_transforms(df: pd.DataFrame, *fns: TransformFn) -> pd.DataFrame:
    """Apply a sequence of transform functions left-to-right."""
    return reduce(lambda acc, fn: fn(acc), fns, df)


# ── Run transforms
transformed_df = apply_transforms(
    valid_df,
    drop_duplicates,
    fill_category,
    add_amount_tier,
    extract_date_parts,
    normalize_amount,
)

logger.info(f"Transform complete. Columns: {transformed_df.columns.tolist()}")
print(f"\nFinal shape: {transformed_df.shape}")
print(f"Tier distribution:\n{transformed_df['tier'].value_counts()}")
transformed_df.head()

06:00:21 | INFO     | Transform complete. Columns: ['id', 'amount', 'category', 'ts', 'tier', 'date', 'hour', 'day_of_week', 'amount_z']

Final shape: (989, 9)
Tier distribution:
tier
high       452
medium     441
low         50
premium     46
Name: count, dtype: int64


,id,amount,category,ts,tier,date,hour,day_of_week,amount_z
0,0,109.14,A,2024-01-01 00:00:00,high,2024-01-01,0,Monday,0.3298
1,1,68.80,A,2024-01-01 01:00:00,medium,2024-01-01,1,Monday,-1.0421
2,2,122.51,C,2024-01-01 02:00:00,high,2024-01-01,2,Monday,0.7846
3,3,128.22,C,2024-01-01 03:00:00,high,2024-01-01,3,Monday,0.9788
4,4,41.47,A,2024-01-01 04:00:00,low,2024-01-01,4,Monday,-1.9716


Quality Gate & Alerting

In [ ]:
# Cell [7] — Quality gate with structured checks and alerting hook
from dataclasses import dataclass as dc
from typing import List


@dc
class QualityCheck:
    name: str
    passed: bool
    value: float
    threshold: float
    description: str = ""

    def __str__(self):
        icon = "✓" if self.passed else "✗"
        return (
            f"  {icon} {self.name:<20} "
            f"value={self.value:.4f}  threshold={self.threshold}"
        )


def send_alert(to: str, subject: str, body: str) -> None:
    """Alert dispatcher. Replace with SMTP / Slack / PagerDuty in production."""
    logger.warning(f"ALERT → {to} | {subject} | {body}")
    # Example Slack webhook:
    # import requests
    # requests.post(SLACK_WEBHOOK, json={"text": f"*{subject}*\n{body}"})


def run_quality_gate(df: pd.DataFrame, cfg: PipelineConfig) -> List[QualityCheck]:
    """Run all quality checks. Raises RuntimeError on any failure."""
    checks: List[QualityCheck] = [
        QualityCheck(
            name="null_rate",
            passed=df.isnull().mean().max() < cfg.null_rate_limit,
            value=float(df.isnull().mean().max()),
            threshold=cfg.null_rate_limit,
            description="Max null rate across all columns",
        ),
        QualityCheck(
            name="row_count",
            passed=len(df) >= cfg.min_rows,
            value=float(len(df)),
            threshold=float(cfg.min_rows),
            description="Minimum row count",
        ),
        QualityCheck(
            name="amount_mean",
            passed=80 <= df["amount"].mean() <= 120,
            value=float(df["amount"].mean()),
            threshold=100.0,
            description="Amount mean within expected range [80, 120]",
        ),
        QualityCheck(
            name="dup_id_rate",
            passed=float(df["id"].duplicated().mean()) < 0.01,
            value=float(df["id"].duplicated().mean()),
            threshold=0.01,
            description="Duplicate ID rate below 1%",
        ),
        QualityCheck(
            name="amount_z_range",
            passed=df["amount_z"].abs().max() < 5.0,
            value=float(df["amount_z"].abs().max()),
            threshold=5.0,
            description="No extreme outliers (|z| < 5)",
        ),
    ]

    print("Quality Gate Results:")
    for c in checks:
        level = "info" if c.passed else "error"
        getattr(logger, level)(str(c))

    failures = [c for c in checks if not c.passed]
    if failures:
        names = [f.name for f in failures]
        msg = f"Quality gate FAILED on: {names}"
        logger.error(msg)
        send_alert(
            to=cfg.alert_email,
            subject="Pipeline Quality Gate Failure",
            body=msg,
        )
        raise RuntimeError(msg)

    logger.info("Quality gate PASSED — all checks green")
    return checks


# ── Run quality gate
qc_results = run_quality_gate(transformed_df, cfg)

Quality Gate Results:
06:00:54 | INFO     |   ✓ null_rate            value=0.0000  threshold=0.05
06:00:54 | INFO     |   ✓ row_count            value=989.0000  threshold=100.0
06:00:54 | INFO     |   ✓ amount_mean          value=99.4414  threshold=100.0
06:00:54 | INFO     |   ✓ dup_id_rate          value=0.0000  threshold=0.01
06:00:54 | INFO     |   ✓ amount_z_range       value=3.2625  threshold=5.0
06:00:54 | INFO     | Quality gate PASSED — all checks green


Storage

In [ ]:
# Cell [8] — Store validated & transformed data as partitioned Parquet
import os
from datetime import date


def store_parquet(df: pd.DataFrame, cfg: PipelineConfig) -> Path:
    """Write DataFrame to a date-partitioned Parquet file.

    Output path: {output_path}/date={today}/data.parquet
    """
    today = date.today().isoformat()
    partition_dir = cfg.output_path / f"date={today}"
    partition_dir.mkdir(parents=True, exist_ok=True)

    out_path = partition_dir / "data.parquet"
    df.to_parquet(out_path, index=False, compression="snappy")

    size_kb = out_path.stat().st_size / 1024
    logger.info(f"Stored {len(df):,} rows → {out_path} ({size_kb:.1f} KB)")
    return out_path


# ── Store
output_path = store_parquet(transformed_df, cfg)

# Verify round-trip
readback = pd.read_parquet(output_path)
assert len(readback) == len(transformed_df), "Row count mismatch on readback!"
logger.info(f"Round-trip verified: {len(readback):,} rows read back successfully")
print(f"\nOutput: {output_path}")

Full Pipeline Runner (with Retry)

In [ ]:
# Cell [9] — End-to-end pipeline runner with Tenacity retry
from tenacity import (
    retry,
    stop_after_attempt,
    wait_exponential,
    retry_if_exception_type,
    before_sleep_log,
)
import logging
import os
from datetime import date
from pathlib import Path
import pandas as pd # Ensure pandas is imported if store_parquet is moved here


_std_logger = logging.getLogger("pipeline")


def store_parquet(df: pd.DataFrame, cfg: PipelineConfig) -> Path:
    """Write DataFrame to a date-partitioned Parquet file.

    Output path: {output_path}/date={today}/data.parquet
    """
    today = date.today().isoformat()
    partition_dir = cfg.output_path / f"date={today}"
    partition_dir.mkdir(parents=True, exist_ok=True)

    out_path = partition_dir / "data.parquet"
    df.to_parquet(out_path, index=False, compression="snappy")

    size_kb = out_path.stat().st_size / 1024
    logger.info(f"Stored {len(df):,} rows → {out_path} ({size_kb:.1f} KB)")
    return out_path


@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=4, max=60),
    retry=retry_if_exception_type((IOError, ConnectionError)),
    before_sleep=before_sleep_log(_std_logger, logging.WARNING),
    reraise=True,
)
def run_pipeline(cfg: PipelineConfig) -> dict:
    """Execute the full ETL pipeline end-to-end.

    Stages:
        1. Ingest      — load raw data from source
        2. Validate    — Pydantic row-level schema check
        3. GE suite    — dataset-level expectation checks
        4. Transform   — enrich, normalise, bin
        5. Quality gate— statistical checks, alert on failure
        6. Store       — write partitioned Parquet

    Returns:
        dict with run statistics
    """
    logger.info("=" * 50)
    logger.info("Pipeline run STARTED")
    logger.info("=" * 50)

    try:
        # 1. Ingest
        raw = SyntheticIngestor(n=cfg.batch_size).read()

        # 2. Pydantic validation
        valid, errors = validate_batch(raw, cfg.pass_rate_threshold)

        # 3. Great Expectations suite
        ge = build_ge_suite(valid)
        if not ge["success"]:
            raise RuntimeError("Great Expectations suite failed")

        # 4. Transforms
        processed = apply_transforms(
            valid,
            drop_duplicates,
            fill_category,
            add_amount_tier,
            extract_date_parts,
            normalize_amount,
        )

        # 5. Quality gate
        run_quality_gate(processed, cfg)

        # 6. Store
        out = store_parquet(processed, cfg)

        stats = {
            "status": "success",
            "raw_rows": len(raw),
            "valid_rows": len(valid),
            "validation_errors": len(errors),
            "output_path": str(out),
        }
        logger.info("=" * 50)
        logger.info("Pipeline run SUCCEEDED")
        logger.info(f"Stats: {stats}")
        logger.info("=" * 50)
        return stats

    except Exception as e:
        logger.error(f"Pipeline FAILED: {e}")
        send_alert(cfg.alert_email, "Pipeline Failure", str(e))
        raise


# ── Run pipeline once immediately
run_stats = run_pipeline(cfg)
print(f"\nRun stats: {run_stats}")

06:02:25 | INFO     | ==================================================
06:02:25 | INFO     | Pipeline run STARTED
06:02:25 | INFO     | ==================================================
06:02:25 | INFO     | Generated 10,000 synthetic rows (seed=42)
06:02:25 | INFO     | Validation: 9,896 valid | 104 errors | 99.0% pass rate


INFO:great_expectations.data_context.types.base:Created temporary directory '/tmp/tmpgh_1y8en' for ephemeral docs site


Calculating Metrics:   0%|          | 0/25 [00:00<?, ?it/s]

06:02:25 | INFO     | GE suite: 6/6 expectations passed
Quality Gate Results:
06:02:25 | INFO     |   ✓ null_rate            value=0.0000  threshold=0.05
06:02:25 | INFO     |   ✓ row_count            value=9896.0000  threshold=100.0
06:02:25 | INFO     |   ✓ amount_mean          value=99.7009  threshold=100.0
06:02:25 | INFO     |   ✓ dup_id_rate          value=0.0000  threshold=0.01
06:02:25 | INFO     |   ✓ amount_z_range       value=4.0192  threshold=5.0
06:02:25 | INFO     | Quality gate PASSED — all checks green
06:02:25 | INFO     | Stored 9,896 rows → data/processed/date=2026-06-05/data.parquet (257.3 KB)
06:02:25 | INFO     | ==================================================
06:02:25 | INFO     | Pipeline run SUCCEEDED
06:02:25 | INFO     | Stats: {'status': 'success', 'raw_rows': 10000, 'valid_rows': 9896, 'validation_errors': 104, 'output_path': 'data/processed/date=2026-06-05/data.parquet'}
06:02:25 | INFO     | ==================================================

Run stats

Scheduling with APScheduler

In [ ]:
# Cell [10] — APScheduler cron trigger + graceful shutdown
# NOTE: In Colab, this cell starts a background scheduler.
# Run the cell below to stop it cleanly.
from apscheduler.schedulers.background import BackgroundScheduler
from apscheduler.triggers.cron import CronTrigger
import signal, atexit

scheduler = BackgroundScheduler(timezone="UTC")

scheduler.add_job(
    run_pipeline,
    trigger=CronTrigger.from_crontab(cfg.schedule_cron),
    kwargs={"cfg": cfg},
    id="daily_pipeline",
    max_instances=1,        # prevent overlapping runs
    coalesce=True,          # run once even if multiple triggers missed
    misfire_grace_time=3600,  # tolerate up to 1-hour misfire
    replace_existing=True,
)

# Clean shutdown on process exit
atexit.register(lambda: scheduler.shutdown(wait=False))

scheduler.start()

jobs = scheduler.get_jobs()
logger.info(f"Scheduler started | {len(jobs)} job(s) registered")
for job in jobs:
    logger.info(f"  Job '{job.id}' | next run: {job.next_run_time}")

print("\nScheduler is running in the background.")
print(f"Cron: {cfg.schedule_cron} UTC")
print("Run the next cell to stop it.")

06:02:36 | INFO     | Scheduler started | 1 job(s) registered
06:02:36 | INFO     |   Job 'daily_pipeline' | next run: 2026-06-06 06:00:00+00:00

Scheduler is running in the background.
Cron: 0 6 * * * UTC
Run the next cell to stop it.


In [ ]:
# Cell [11] — Stop the scheduler (run this before kernel shutdown)
if scheduler.running:
    scheduler.shutdown(wait=False)
    logger.info("Scheduler stopped cleanly")
else:
    logger.info("Scheduler was not running")

06:02:53 | INFO     | Scheduler stopped cleanly


Extension C · Prometheus Metrics + Grafana Dashboard

In [ ]:
# Prometheus Metrics

from prometheus_client import (
    Counter,
    Histogram,
    Gauge,
    start_http_server
)

import time
from contextlib import contextmanager

start_http_server(8000) #Start metrics endpoint

PIPELINE_ROWS = Counter(                     # Total rows processed by stage
    "pipeline_rows_total",
    "Number of rows processed",
    ["stage"]
)

STAGE_DURATION_SECONDS = Histogram(           # Stage execution duration
    "stage_duration_seconds",
    "Execution duration per pipeline stage",
    ["stage"],
    buckets = (0.1, 0.5, 1, 2, 5, 10, 30, 60, 120, 300)
)

VALIDATION_PASS_RATE = Gauge(
    "validation_pass_rate",
    "Percentage of rows passing validation"
)

PIPELINE_RUNS_TOTAL = Counter(
    "pipeline_runs_total",
    "Pipeline executions",
    ["status"]
)


PIPELINE_ERRORS_TOTAL = Counter(
    "pipeline_errors_total",
    "Pipeline errors",
    ["stage"]
)


In [ ]:
# Stage Timer Decorator
@contextmanager
def track_stage(stage_name):
    start = time.perf_counter()

    try:
        yield
    except Exception:
        PIPELINE_ERRORS_TOTAL.labels(stage=stage_name).inc()
        raise
    finally:
        STAGE_DURATION_SECONDS.labels(
            stage=stage_name
        ).observe(time.perf_counter() - start)

In [ ]:
# Instrument Each Pipeline Stage
def run_pipeline(cfg):

    try:
        with track_stage("ingest"):
            df = ingestor.read()
            PIPELINE_ROWS.labels(stage="ingest").inc(len(df))

        with track_stage("validate"):
            valid_df, invalid_df = validate_dataframe(df)

            PIPELINE_ROWS.labels(
                stage="validate_pass"
            ).inc(len(valid_df))

            PIPELINE_ROWS.labels(
                stage="validate_fail"
            ).inc(len(invalid_df))

            pass_rate = len(valid_df) / len(df)
            VALIDATION_PASS_RATE.set(pass_rate)

        with track_stage("transform"):
            transformed_df = transform(valid_df)
            PIPELINE_ROWS.labels(
                stage="transform"
            ).inc(len(transformed_df))

        with track_stage("quality_gate"):
            run_quality_gate(transformed_df, cfg)

        with track_stage("store"):
            store_parquet(transformed_df, cfg)

        PIPELINE_RUNS_TOTAL.labels(
            status="success"
        ).inc()

    except Exception:
        PIPELINE_RUNS_TOTAL.labels(
            status="failure"
        ).inc()
        raise

In [ ]:
# Test the metrics
import requests

response = requests.get("http://localhost:8000/metrics")
print(response.text[:2000])  # first 2000 chars

# HELP python_gc_objects_collected_total Objects collected during gc
# TYPE python_gc_objects_collected_total counter
python_gc_objects_collected_total{generation="0"} 57148.0
python_gc_objects_collected_total{generation="1"} 8849.0
python_gc_objects_collected_total{generation="2"} 3325.0
# HELP python_gc_objects_uncollectable_total Uncollectable objects found during GC
# TYPE python_gc_objects_uncollectable_total counter
python_gc_objects_uncollectable_total{generation="0"} 0.0
python_gc_objects_uncollectable_total{generation="1"} 0.0
python_gc_objects_uncollectable_total{generation="2"} 0.0
# HELP python_gc_collections_total Number of times this generation was collected
# TYPE python_gc_collections_total counter
python_gc_collections_total{generation="0"} 2064.0
python_gc_collections_total{generation="1"} 187.0
python_gc_collections_total{generation="2"} 12.0
# HELP python_info Python platform information
# TYPE python_info gauge
python_info{implementation="CPython",major="3",minor="

# alerts.yml
groups:
  - name: pipeline-alerts
    rules:
      - alert: ValidationPassRateLow
        expr: validation_pass_rate < 0.95
        for: 5m

In [ ]:
%%capture
!pip install pyyaml

In [ ]:
# Validate YAML

import yaml

# Create the alerts.yml file with the specified content
# This content was previously defined in text cell CI2jIBIEdcgW
alerts_yaml_content = """
groups:
  - name: pipeline-alerts
    rules:
      - alert: ValidationPassRateLow
        expr: validation_pass_rate < 0.95
        for: 5m
"""

with open("alerts.yml", "w") as f:
    f.write(alerts_yaml_content)

print("alerts.yml created successfully.")

with open("alerts.yml") as f:
    data = yaml.safe_load(f)

print("YAML is valid")


alerts.yml created successfully.
YAML is valid
